# Insurance Claims Data Model — Exploration & SQL Execution Notebook

This notebook demonstrates how the data model works in practice by:
- Loading synthetic datasets  
- Creating an in‑memory SQLite database  
- Executing the physical SQL schema  
- Inserting sample data  
- Running analytical SQL queries  
- Displaying results  

This mirrors the workflow of a Junior Data Modeller validating a schema.

---

## 1. Setup

Import required libraries and create a connection to an in‑memory SQLite database.

In [ ]:
import sqlite3
import pandas as pd

# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

## 2. Load Synthetic Data

Load the CSV files from the `data/synthetic/` folder into pandas DataFrames.

In [ ]:
customers = pd.read_csv("../data/synthetic/customers.csv")
policies = pd.read_csv("../data/synthetic/policies.csv")
claims = pd.read_csv("../data/synthetic/claims.csv")
payments = pd.read_csv("../data/synthetic/payments.csv")
assessors = pd.read_csv("../data/synthetic/assessors.csv")
assessments = pd.read_csv("../data/synthetic/assessments.csv")

customers.head()


## 3. Create Database Schema

Execute the SQL schema from `models/physical/schema.sql`.

In [ ]:
with open("../models/physical/schema.sql", "r") as f:
    schema_sql = f.read()

cursor.executescript(schema_sql)
conn.commit()


## 4. Insert Synthetic Data into Tables

In [ ]:
customers.to_sql("Customer", conn, if_exists="append", index=False)
policies.to_sql("Policy", conn, if_exists="append", index=False)
claims.to_sql("Claim", conn, if_exists="append", index=False)
payments.to_sql("Payment", conn, if_exists="append", index=False)
assessors.to_sql("Assessor", conn, if_exists="append", index=False)
assessments.to_sql("Assessment", conn, if_exists="append", index=False)


## 5. Run Sample SQL Queries

These queries demonstrate how the model supports reporting and analysis.

In [ ]:
# 5.1 Claims per customer
query = """
SELECT 
    c.CustomerID,
    c.FirstName,
    c.LastName,
    COUNT(cl.ClaimID) AS TotalClaims
FROM Customer c
JOIN Policy p ON c.CustomerID = p.CustomerID
JOIN Claim cl ON p.PolicyID = cl.PolicyID
GROUP BY c.CustomerID, c.FirstName, c.LastName;
"""

pd.read_sql_query(query, conn)


In [ ]:
# 5.2 Total payments per claim
query = """
SELECT 
    ClaimID,
    SUM(PaymentAmount) AS TotalPaid
FROM Payment
GROUP BY ClaimID;
"""

pd.read_sql_query(query, conn)


In [ ]:
# 5.3 Claims by status
query = """
SELECT 
    ClaimStatus,
    COUNT(*) AS Count
FROM Claim
GROUP BY ClaimStatus;
"""

pd.read_sql_query(query, conn)


In [ ]:
# 5.4 Assessments per assessor
query = """
SELECT 
    a.AssessorID,
    a.Name,
    COUNT(asmt.AssessmentID) AS TotalAssessments
FROM Assessor a
LEFT JOIN Assessment asmt ON a.AssessorID = asmt.AssessorID
GROUP BY a.AssessorID, a.Name;
"""

pd.read_sql_query(query, conn)


## 6. Explore the Data Model Interactively

Use pandas to inspect joins and relationships visually.

In [ ]:
merged = claims.merge(policies, on="PolicyID").merge(customers, on="CustomerID")
merged.head()

## 7. Conclusion

This notebook validates the insurance claims data model by:
- Successfully loading structured data  
- Creating a relational schema  
- Enforcing relationships  
- Running analytical queries  
- Demonstrating real‑world usage  

This mirrors the workflow of a Junior Data Modeller working with business stakeholders and data engineers to ensure data structures support reporting and operational needs.